[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CS7150/classdemos/blob/main/optimization/adam-invariance.ipynb)

# SGD vs. momentum vs. Adam, rescaled

Companion notebook to the [interactive demo](https://cs7150.github.io/classdemos/demos/optimizers/adam-invariance.html).

### Why does rescaling the gradient matter?

Suppose we multiply every gradient $g$ we ever compute by some constant $K$ -- maybe the loss
was defined with an extra factor of 10, or the data was rescaled, or a different (but
equivalent) parameterization happens to produce gradients 100x larger. A good optimizer
shouldn't care: the *loss surface* hasn't changed, only the units we're measuring its slope in.
But whether an optimizer actually is indifferent to $K$ depends entirely on its update rule.

**Plain SGD** takes a step proportional to the gradient itself:

$$x \leftarrow x - \eta\, g.$$

If $g$ is replaced by $Kg$, the step becomes $\eta K g$ -- $K$ times too large (or too small).
Nothing in the update cancels it, so the learning rate $\eta$ that was well-tuned for one scale
of gradient can make the optimizer crawl or diverge at another.

**Momentum** replaces the gradient with its exponential moving average,

$$m \leftarrow \beta_1 m + (1-\beta_1) g, \qquad x \leftarrow x - \eta\, m,$$

but averaging is a linear operation: if every $g$ is scaled by $K$, then $m$ is scaled by $K$
too (its recurrence is linear in $g$), and the step $\eta m$ still scales with $K$. Momentum
smooths out noise, but it inherits the same sensitivity to the overall gradient scale that SGD
has.

**Adam** adds a second EMA, of the *squared* gradient, and divides by its square root:

$$m \leftarrow \beta_1 m + (1-\beta_1) g, \qquad v \leftarrow \beta_2 v + (1-\beta_2) g^2,$$

with bias-corrected estimates $\hat m = m / (1-\beta_1^t)$ and $\hat v = v / (1-\beta_2^t)$, and
update

$$x \leftarrow x - \eta\, \frac{\hat m}{\sqrt{\hat v} + \epsilon}.$$

Now watch what happens under $g \to Kg$: since $m$ is linear in $g$, $\hat m \to K\hat m$.
Since $v$ is an EMA of $g^2$, $v \to K^2 v$, so $\sqrt{\hat v} \to K\sqrt{\hat v}$ (taking
$K>0$). The ratio $\hat m / \sqrt{\hat v}$ is therefore *unchanged* -- the factor of $K$ appears
identically in the numerator and denominator and cancels exactly (up to the small constant
$\epsilon$, which matters only when gradients are tiny). Adam's step size depends on the
*direction and relative noisiness* of the gradient, not on its absolute scale.

The notebook below runs all three optimizers on the same tilted, anisotropic quadratic bowl,
first with $K=1$ and then with $K=10$, so you can see this cancellation happen in practice:
SGD's and momentum's loss curves shift dramatically between the two runs, while Adam's are
nearly identical.

In [ ]:
#@title Setup: loss landscape + plotting helper (double-click to inspect) { display-mode: "form" }
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# Same anisotropic, tilted bowl as the interactive demo: a rotated quadratic
# loss(x, y) = 0.5 * (Q00 x^2 + 2 Q01 xy + Q11 y^2), with curvature ratio `a`
# along its steep axis vs. 1 along its shallow axis, tilted by `theta_deg`.
a = 20.0
theta_deg = -25.0

def make_Q(a=a, theta_deg=theta_deg):
    th = np.radians(theta_deg)
    c, s = np.cos(th), np.sin(th)
    Q00 = a * s * s + c * c
    Q11 = a * c * c + s * s
    Q01 = (1 - a) * s * c
    return np.array([[Q00, Q01], [Q01, Q11]])

Q = make_Q()

def loss(xy):
    return 0.5 * xy @ Q @ xy

def grad(xy):
    return Q @ xy

def start_point():
    # Same construction as the demo: start well out along the shallow axis.
    evals, evecs = np.linalg.eigh(Q)
    steep, shallow = evecs[:, 1], evecs[:, 0]  # eigh sorts ascending
    return 0.4 * steep + 1.4 * shallow

def plot_losses(loss_histories, labels, colors, title):
    fig, ax = plt.subplots(figsize=(5, 3.5))
    for hist, label, color in zip(loss_histories, labels, colors):
        ax.plot(hist, color=color, label=label, linewidth=2)
    ax.set_yscale('log')
    ax.set_xlabel('iteration')
    ax.set_ylabel('loss (log scale)')
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    return fig, ax


## The three update rules, run for `K = 1`

In [ ]:
eta = 0.1
beta1, beta2, eps = 0.9, 0.999, 1e-8
n_iters = 90

def run_sgd(K, n_iters=n_iters):
    x = start_point()
    losses = [loss(x)]
    for t in range(1, n_iters + 1):
        g = K * grad(x)
        x = x - eta * g
        losses.append(loss(x))
    return losses

def run_momentum(K, n_iters=n_iters):
    x = start_point()
    m = np.zeros(2)
    losses = [loss(x)]
    for t in range(1, n_iters + 1):
        g = K * grad(x)
        m = beta1 * m + (1 - beta1) * g
        x = x - eta * m
        losses.append(loss(x))
    return losses

def run_adam(K, n_iters=n_iters):
    x = start_point()
    m = np.zeros(2)
    v = np.zeros(2)
    losses = [loss(x)]
    for t in range(1, n_iters + 1):
        g = K * grad(x)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        x = x - eta * m_hat / (np.sqrt(v_hat) + eps)
        losses.append(loss(x))
    return losses

losses_sgd_K1 = run_sgd(K=1.0)
losses_mom_K1 = run_momentum(K=1.0)
losses_adam_K1 = run_adam(K=1.0)


## Now rescale every gradient by `K = 10` and run the same three loops

In [ ]:
losses_sgd_K10 = run_sgd(K=10.0)
losses_mom_K10 = run_momentum(K=10.0)
losses_adam_K10 = run_adam(K=10.0)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)

for ax, (K, hists) in zip(axes, [
    (1, (losses_sgd_K1, losses_mom_K1, losses_adam_K1)),
    (10, (losses_sgd_K10, losses_mom_K10, losses_adam_K10)),
]):
    labels = ['SGD', 'momentum', 'Adam']
    colors = ['tab:orange', 'tab:green', 'tab:blue']
    for hist, label, color in zip(hists, labels, colors):
        ax.plot(hist, color=color, label=label, linewidth=2)
    ax.set_yscale('log')
    ax.set_xlabel('iteration')
    ax.set_title(f'K = {K}')
    ax.legend()
axes[0].set_ylabel('loss (log scale)')
fig.suptitle('SGD and momentum blow up when K grows; Adam barely notices')
fig.tight_layout()
plt.show()


**What to notice:** going from the left panel (`K=1`) to the right panel
(`K=10`), the SGD and momentum curves shift dramatically -- with `eta=0.1`
tuned for `K=1`, a 10x larger gradient makes their step sizes 10x too big,
so they diverge or oscillate wildly. Adam's curve is nearly unchanged,
because its update divides `m` by `sqrt(v)`, and scaling every gradient by
`K` scales both `m` and `sqrt(v)` by the same factor `K`, which cancels.